真实的哈特曼传感器程序

In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np

# SI units: m, rad, px, unless the key name says otherwise.
PROJECT_DIR = Path.cwd()

params = {
    "source": {
        "wavelength_m": 561e-9,
    },
    "camera": {
        "resolution_px": (1440, 1080),
        "pixel_pitch_m": 3.45e-6,
    },
    "microlens_array": {
        "model": "MLAS10-F15-P300-AB",
        "pitch_m": 300e-6,
        "focal_length_m": 14.6e-3,
    },
    "relay": {
        "dm_to_sh_magnification": 1.0,
    },
    "dm": {
        "sdk_dir": str(PROJECT_DIR / "python3.9_bmc"),
        "serial_number": "17DW013#137",
        "flat_map_file": str(PROJECT_DIR / "17DW013#137_FLAT_MAP_COMMANDS.txt"),
        "active_actuator_array": (12, 12),
        "available_actuators": 140,
        "active_aperture_m": 4.4e-3,
        "stroke_m": 3500e-9,
        "command_range": (0.0, 1.0),
        "flat_command": 0.5,
    },
}

camera_size_m = np.array(params["camera"]["resolution_px"]) * params["camera"]["pixel_pitch_m"]
dm_image_size_m = params["dm"]["active_aperture_m"] * params["relay"]["dm_to_sh_magnification"]

derived_params = {
    "camera_size_mm": tuple(camera_size_m * 1e3),
    "lenslet_pitch_px": params["microlens_array"]["pitch_m"] / params["camera"]["pixel_pitch_m"],
    "dm_image_size_mm": dm_image_size_m * 1e3,
    "dm_image_size_px": dm_image_size_m / params["camera"]["pixel_pitch_m"],
    "dm_lenslets_across": dm_image_size_m / params["microlens_array"]["pitch_m"],
}

pprint(params)
print("\nDerived:")
pprint(derived_params)


加载dm可变镜的控制库

In [ ]:
import os
import sys


def load_bmc_sdk(sdk_dir):
    sdk_dir = Path(sdk_dir)
    if not sdk_dir.exists():
        raise FileNotFoundError(f"BMC SDK directory not found: {sdk_dir}")

    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(sdk_dir))
    if str(sdk_dir) not in sys.path:
        sys.path.insert(0, str(sdk_dir))

    import bmc

    return bmc


def load_flat_commands(flat_map_file, expected_count, command_range):
    flat_map_file = Path(flat_map_file)
    if not flat_map_file.exists():
        raise FileNotFoundError(f"Flat map file not found: {flat_map_file}")

    commands = np.loadtxt(flat_map_file, dtype=np.float64).reshape(-1)
    if commands.size != expected_count:
        raise ValueError(f"Expected {expected_count} flat commands, got {commands.size}.")
    if not np.all(np.isfinite(commands)):
        raise ValueError("Flat commands contain non-finite values.")

    lo, hi = command_range
    if np.any((commands < lo) | (commands > hi)):
        raise ValueError(f"Flat commands must stay in [{lo}, {hi}].")
    return commands


bmc = load_bmc_sdk(params["dm"]["sdk_dir"])
flat_commands = load_flat_commands(
    params["dm"]["flat_map_file"],
    expected_count=params["dm"]["available_actuators"],
    command_range=params["dm"]["command_range"],
)

print(f"BMC SDK version: {bmc.BmcDm.version_string()}")
print(f"Flat map length: {flat_commands.size}")
print(f"Flat map command range: {flat_commands.min():.6f} to {flat_commands.max():.6f}")
print(f"Flat map mean command: {flat_commands.mean():.6f}")


将dm控制函数进行包装

In [ ]:
class BmcDmController:
    def __init__(self, bmc_module, dm_params, flat_commands):
        self.bmc = bmc_module
        self.dm_params = dm_params
        self.flat_commands = np.asarray(flat_commands, dtype=np.float64)
        self.dm = None

    def _check(self, code, action):
        if int(code) != int(self.bmc.NO_ERR):
            message = ""
            if self.dm is not None:
                message = self.dm.error_string(int(code))
            raise RuntimeError(f"{action} failed: code {code}. {message}")

    def open(self):
        serial_number = self.dm_params["serial_number"]
        if not serial_number:
            raise ValueError("Set params['dm']['serial_number'] before opening the DM.")

        self.dm = self.bmc.BmcDm()
        self._check(self.dm.open_dm(serial_number), "open_dm")

        actuator_count = int(self.dm.num_actuators())
        if actuator_count != self.flat_commands.size:
            self.close()
            raise RuntimeError(
                f"DM reports {actuator_count} actuators, but flat map has {self.flat_commands.size}."
            )

        print(f"Opened DM {serial_number}: {actuator_count} actuators")
        return self

    def send(self, commands, label="commands"):
        if self.dm is None:
            raise RuntimeError("Open the DM before sending commands.")

        commands = np.asarray(commands, dtype=np.float64).reshape(-1)
        if commands.size != self.flat_commands.size:
            raise ValueError(f"Expected {self.flat_commands.size} commands, got {commands.size}.")
        if not np.all(np.isfinite(commands)):
            raise ValueError(f"{label} contain non-finite values.")

        lo, hi = self.dm_params["command_range"]
        if np.any((commands < lo) | (commands > hi)):
            raise ValueError(f"{label} must stay in [{lo}, {hi}].")

        self._check(self.dm.send_data(self.bmc.DoubleVector(commands.tolist())), f"send {label}")
        return commands

    def send_flat(self):
        sent = self.send(self.flat_commands, label="flat map")
        actual = np.asarray(list(self.dm.get_actuator_data()), dtype=np.float64)
        print(f"Flat map sent. Stored-command max error: {np.max(np.abs(actual - sent)):.3e}")
        return sent

    def close(self):
        if self.dm is not None:
            code = self.dm.close_dm()
            self._check(code, "close_dm")
            self.dm = None
            print("DM closed. Note: BMC close_dm sets output to zero, not to the flat map.")


运行下面这个单元会打开真实 DM，并发送厂家 flat map，产生纯平平面。

In [ ]:
dm_ctrl = BmcDmController(bmc, params["dm"], flat_commands).open()
dm_ctrl.send_flat()


采集初始哈特曼图像的一些准备工作

In [ ]:
import time
from datetime import datetime

import matplotlib.pyplot as plt


reference_capture_params = {
    "camera_index": 1,
    "exposure_us": 5000.0,
    "gain": 0.0,
    "aoi_px": (0, 0, 2592, 1944),
    "warmup_frames": 2,
    "settle_s": 0.2,
    "save_dir": PROJECT_DIR / "real_captures",
}


def load_daheng_camera_class():
    try:
        from camera import _GX
    except ModuleNotFoundError as exc:
        if exc.name == "gxipy":
            raise ModuleNotFoundError(
                "The Daheng gxipy package is not available in this Python environment. "
                "Install Daheng's Galaxy SDK Python package into liteloc_env first."
            ) from exc
        raise
    return _GX


def show_sh_image(image, title="Shack-Hartmann image", output_png=None):
    image = np.asarray(image)
    vmin, vmax = np.percentile(image, [1.0, 99.8])
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(image, cmap="gray", origin="upper", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if output_png is not None:
        fig.savefig(output_png, dpi=200)
    plt.show()


def capture_reference_image(capture_params):
    _GX = load_daheng_camera_class()
    save_dir = Path(capture_params["save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)

    camera = None
    try:
        camera = _GX(
            camera_index=capture_params["camera_index"],
            exposure_us=capture_params["exposure_us"],
            gain=capture_params["gain"],
        )
        camera.stream_off()
        camera.setAOI(*capture_params["aoi_px"])
        camera.stream_on()
        time.sleep(float(capture_params.get("settle_s", 0.0)))

        for _ in range(int(capture_params.get("warmup_frames", 0))):
            camera.captureImage()
            time.sleep(0.05)

        image = np.asarray(camera.captureImage())
    finally:
        if camera is not None:
            camera.shutDown()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    npy_path = save_dir / f"reference_flat_{timestamp}.npy"
    png_path = save_dir / f"reference_flat_{timestamp}.png"
    np.save(npy_path, image)
    show_sh_image(image, title="Initial reference image", output_png=png_path)

    print(f"Reference image shape: {image.shape}, dtype: {image.dtype}")
    print(f"Saved NPY: {npy_path}")
    print(f"Saved preview PNG: {png_path}")
    return image, {"npy": npy_path, "png": png_path}


运行下面这个单元会调用大恒相机采集一张初始 reference 图像。

In [ ]:
reference_image, reference_image_paths = capture_reference_image(reference_capture_params)


检测初始图像中的质点位置

In [ ]:
import pandas as pd
from scipy import ndimage as ndi


spot_detection_params = {
    "background_percentile": 5.0,
    "blur_sigma_px": 1.0,
    "threshold_relative": 0.20,
    "min_area_px": 5,
    "max_area_px": 800,
    "edge_margin_px": 2,
}


def detect_spot_centroids(image, detection_params):
    image = np.asarray(image, dtype=np.float64)
    background = np.percentile(image, detection_params["background_percentile"])
    signal = np.clip(image - background, 0.0, None)

    if detection_params["blur_sigma_px"] > 0:
        filtered = ndi.gaussian_filter(signal, sigma=detection_params["blur_sigma_px"])
    else:
        filtered = signal

    peak = float(np.max(filtered))
    if peak <= 0:
        raise ValueError("No positive signal found in the reference image.")

    threshold = detection_params["threshold_relative"] * peak
    labels, n_labels = ndi.label(filtered >= threshold, structure=np.ones((3, 3), dtype=bool))
    slices = ndi.find_objects(labels)

    records = []
    h, w = image.shape
    edge_margin = int(detection_params["edge_margin_px"])

    for label_id, slc in enumerate(slices, start=1):
        if slc is None:
            continue

        yy0, yy1 = slc[0].start, slc[0].stop
        xx0, xx1 = slc[1].start, slc[1].stop
        component = labels[slc] == label_id
        area = int(np.count_nonzero(component))
        if area < detection_params["min_area_px"] or area > detection_params["max_area_px"]:
            continue
        if xx0 < edge_margin or yy0 < edge_margin or xx1 > w - edge_margin or yy1 > h - edge_margin:
            continue

        yy_local, xx_local = np.nonzero(component)
        yy = yy_local + yy0
        xx = xx_local + xx0
        weights = signal[yy, xx]
        weight_sum = float(np.sum(weights))
        if weight_sum <= 0:
            weights = np.ones_like(weights, dtype=np.float64)
            weight_sum = float(weights.size)

        x_px = float(np.sum(xx * weights) / weight_sum)
        y_px = float(np.sum(yy * weights) / weight_sum)
        records.append(
            {
                "x_px": x_px,
                "y_px": y_px,
                "area_px": area,
                "peak_value": float(np.max(image[yy, xx])),
                "sum_signal": weight_sum,
            }
        )

    spots = pd.DataFrame.from_records(records)
    if not spots.empty:
        spots = spots.sort_values(["y_px", "x_px"], ignore_index=True)
        spots.insert(0, "spot_id", np.arange(spots.shape[0], dtype=int))
    return spots, {"threshold": threshold, "background": background, "n_labels": n_labels}


def plot_detected_spots(image, spots, title="Detected reference spots", output_png=None):
    image = np.asarray(image)
    vmin, vmax = np.percentile(image, [1.0, 99.8])
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image, cmap="gray", origin="upper", vmin=vmin, vmax=vmax)
    if not spots.empty:
        ax.scatter(spots["x_px"], spots["y_px"], s=28, facecolors="none", edgecolors="cyan", linewidths=1.0)
    ax.set_title(f"{title}: {len(spots)} spots")
    ax.axis("off")
    plt.tight_layout()
    if output_png is not None:
        fig.savefig(output_png, dpi=200)
    plt.show()


将初始图像进行保存

In [ ]:
if "reference_image" not in globals():
    raise RuntimeError("Run the reference-image capture cell first, or set reference_image = np.load(...).")

reference_spots, reference_detection_info = detect_spot_centroids(reference_image, spot_detection_params)

spot_csv_path = Path(reference_image_paths["npy"]).with_name(Path(reference_image_paths["npy"]).stem + "_spots.csv")
spot_png_path = Path(reference_image_paths["npy"]).with_name(Path(reference_image_paths["npy"]).stem + "_spots.png")
reference_spots.to_csv(spot_csv_path, index=False)
plot_detected_spots(reference_image, reference_spots, output_png=spot_png_path)

print(f"Detected spots: {len(reference_spots)}")
print(f"Detection threshold: {reference_detection_info['threshold']:.3f}")
print(f"Saved spots CSV: {spot_csv_path}")
print(f"Saved spot overlay PNG: {spot_png_path}")
reference_spots.head()


生成单项 Zernike 像差的 DM 命令

In [ ]:
from math import comb, sqrt


zernike_drive_params = {
    "zernike_j": 6,              # 4/6: astigmatism, 5: defocus. Keep in 1..11 for now.
    "delta_command_peak": 0.02,  # Around several tens of nm in the roughly linear region.
    "sign": 1.0,
}

zernike_names = {
    1: "piston",
    2: "tilt y",
    3: "tilt x",
    4: "astigmatism 45",
    5: "defocus",
    6: "astigmatism 0",
    7: "trefoil y",
    8: "coma y",
    9: "coma x",
    10: "trefoil x",
    11: "primary spherical",
}


def j_to_mn(j):
    n = int(np.ceil((-3.0 + np.sqrt(9.0 + 8.0 * j)) / 2.0))
    m = int(2 * j - n * (n + 2))
    return m, n


def radial_polynomial(n, m_abs, rho):
    if (n - m_abs) % 2:
        return np.zeros_like(rho)
    radial = np.zeros_like(rho, dtype=np.float64)
    for s in range((n - m_abs) // 2 + 1):
        c = (-1) ** s * comb(n - s, s) * comb(n - 2 * s, (n - m_abs) // 2 - s)
        radial += c * rho ** (n - 2 * s)
    return radial


def zernike_mode(j, rho, theta):
    m, n = j_to_mn(int(j))
    radial = radial_polynomial(n, abs(m), rho)
    if m == 0:
        return sqrt(n + 1) * radial
    if m > 0:
        return sqrt(2 * (n + 1)) * radial * np.cos(m * theta)
    return sqrt(2 * (n + 1)) * radial * np.sin(abs(m) * theta)


def available_actuator_mask(dm_params):
    rows, cols = dm_params["active_actuator_array"]
    mask = np.ones((rows, cols), dtype=bool)
    mask[0, 0] = False
    mask[0, -1] = False
    mask[-1, 0] = False
    mask[-1, -1] = False
    if int(np.count_nonzero(mask)) != int(dm_params["available_actuators"]):
        raise ValueError("The 12x12 corner mask does not match the expected actuator count.")
    return mask


def command_vector_to_grid(command_vector, dm_params):
    mask = available_actuator_mask(dm_params)
    grid = np.full(mask.shape, np.nan, dtype=np.float64)
    command_vector = np.asarray(command_vector, dtype=np.float64).reshape(-1)
    if command_vector.size != np.count_nonzero(mask):
        raise ValueError(f"Expected {np.count_nonzero(mask)} commands, got {command_vector.size}.")
    grid[mask] = command_vector
    return grid


def make_zernike_delta_commands(dm_params, zernike_j, delta_command_peak=0.02, sign=1.0):
    zernike_j = int(zernike_j)
    if not 1 <= zernike_j <= 11:
        raise ValueError("For this first real-DM test, keep zernike_j in 1..11.")

    rows, cols = dm_params["active_actuator_array"]
    mask = available_actuator_mask(dm_params)
    x = np.linspace(-1.0, 1.0, cols)
    y = np.linspace(-1.0, 1.0, rows)
    xx, yy = np.meshgrid(x, y, indexing="xy")

    rho_raw = np.sqrt(xx**2 + yy**2)
    rho = rho_raw / np.nanmax(rho_raw[mask])
    theta = np.arctan2(yy, xx)

    mode_grid = zernike_mode(zernike_j, rho, theta)
    mode_active = mode_grid[mask].astype(np.float64)
    if zernike_j != 1:
        mode_active -= np.mean(mode_active)

    max_abs = float(np.max(np.abs(mode_active)))
    if max_abs <= 0:
        raise ValueError("Selected Zernike mode is zero on the available actuator grid.")

    normalized_active = mode_active / max_abs
    delta_commands = float(sign) * float(delta_command_peak) * normalized_active
    delta_grid = command_vector_to_grid(delta_commands, dm_params)
    mode_grid[~mask] = np.nan
    return delta_commands, delta_grid, mode_grid


def build_zernike_dm_command(flat_commands, dm_params, zernike_j, delta_command_peak=0.02, sign=1.0):
    delta_commands, delta_grid, mode_grid = make_zernike_delta_commands(
        dm_params=dm_params,
        zernike_j=zernike_j,
        delta_command_peak=delta_command_peak,
        sign=sign,
    )
    command = np.asarray(flat_commands, dtype=np.float64) + delta_commands
    lo, hi = dm_params["command_range"]
    if np.any((command < lo) | (command > hi)):
        raise ValueError("Generated command is outside the allowed DM command range.")

    info = {
        "zernike_j": int(zernike_j),
        "zernike_name": zernike_names.get(int(zernike_j), "unknown"),
        "delta_grid": delta_grid,
        "mode_grid": mode_grid,
        "flat_grid": command_vector_to_grid(flat_commands, dm_params),
        "command_grid": command_vector_to_grid(command, dm_params),
        "delta_min": float(np.min(delta_commands)),
        "delta_max": float(np.max(delta_commands)),
        "command_min": float(np.min(command)),
        "command_max": float(np.max(command)),
    }
    return command, info


def plot_zernike_dm_command(info):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    items = [
        (info["mode_grid"], f"Zernike j={info['zernike_j']}\n{info['zernike_name']}", "RdBu_r"),
        (info["delta_grid"], "Delta command", "RdBu_r"),
        (info["command_grid"], "Flat + delta", "viridis"),
    ]
    for ax, (image, title, cmap) in zip(axes, items):
        im = ax.imshow(image, origin="lower", cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("actuator x")
        ax.set_ylabel("actuator y")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


In [ ]:
zernike_command, zernike_command_info = build_zernike_dm_command(
    flat_commands=flat_commands,
    dm_params=params["dm"],
    **zernike_drive_params,
)

plot_zernike_dm_command(zernike_command_info)
print(f"Zernike j={zernike_command_info['zernike_j']}: {zernike_command_info['zernike_name']}")
print(f"Delta command range: {zernike_command_info['delta_min']:.4f} to {zernike_command_info['delta_max']:.4f}")
print(f"Final command range: {zernike_command_info['command_min']:.4f} to {zernike_command_info['command_max']:.4f}")


运行下面这个单元会把当前 Zernike 命令发送到真实 DM。

In [ ]:
if "dm_ctrl" not in globals() or dm_ctrl.dm is None:
    raise RuntimeError("Run the DM flat/open cell first, then build zernike_command.")
if "zernike_command" not in globals():
    raise RuntimeError("Run the Zernike command generation cell first.")

dm_ctrl.send(zernike_command, label=f"Zernike j={zernike_command_info['zernike_j']}")
print(f"Sent Zernike j={zernike_command_info['zernike_j']} ({zernike_command_info['zernike_name']}) to DM.")


采集添加相位后的哈特曼图像

In [ ]:
def capture_measurement_image(capture_params, command_info=None):
    _GX = load_daheng_camera_class()
    save_dir = Path(capture_params["save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)

    camera = None
    try:
        camera = _GX(
            camera_index=capture_params["camera_index"],
            exposure_us=capture_params["exposure_us"],
            gain=capture_params["gain"],
        )
        camera.stream_off()
        camera.setAOI(*capture_params["aoi_px"])
        camera.stream_on()
        time.sleep(float(capture_params.get("settle_s", 0.0)))

        for _ in range(int(capture_params.get("warmup_frames", 0))):
            camera.captureImage()
            time.sleep(0.05)

        image = np.asarray(camera.captureImage())
    finally:
        if camera is not None:
            camera.shutDown()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if command_info is None:
        prefix = f"measurement_{timestamp}"
        title = "Measurement image"
    else:
        prefix = f"measurement_z{command_info['zernike_j']}_{timestamp}"
        title = f"Measurement image, Zernike j={command_info['zernike_j']}"

    npy_path = save_dir / f"{prefix}.npy"
    png_path = save_dir / f"{prefix}.png"
    np.save(npy_path, image)
    show_sh_image(image, title=title, output_png=png_path)

    print(f"Measurement image shape: {image.shape}, dtype: {image.dtype}")
    print(f"Saved NPY: {npy_path}")
    print(f"Saved preview PNG: {png_path}")
    return image, {"npy": npy_path, "png": png_path}


运行下面这个单元会在当前 DM 相位状态下采集传感器图像。

In [ ]:
if "zernike_command_info" not in globals():
    raise RuntimeError("Run the Zernike command generation/send cells before measurement capture.")

measurement_image, measurement_image_paths = capture_measurement_image(
    reference_capture_params,
    command_info=zernike_command_info,
)


检测 measurement 图像中的质点位置

In [ ]:
if "measurement_image" not in globals():
    raise RuntimeError("Run the measurement-image capture cell first, or set measurement_image = np.load(...).")

measurement_spots, measurement_detection_info = detect_spot_centroids(measurement_image, spot_detection_params)

measurement_spot_csv_path = Path(measurement_image_paths["npy"]).with_name(
    Path(measurement_image_paths["npy"]).stem + "_spots.csv"
)
measurement_spot_png_path = Path(measurement_image_paths["npy"]).with_name(
    Path(measurement_image_paths["npy"]).stem + "_spots.png"
)
measurement_spots.to_csv(measurement_spot_csv_path, index=False)
plot_detected_spots(measurement_image, measurement_spots, title="Detected measurement spots", output_png=measurement_spot_png_path)

print(f"Detected measurement spots: {len(measurement_spots)}")
print(f"Detection threshold: {measurement_detection_info['threshold']:.3f}")
print(f"Saved spots CSV: {measurement_spot_csv_path}")
print(f"Saved spot overlay PNG: {measurement_spot_png_path}")
measurement_spots.head()


定义匹配质点位移并反演 Zernike 系数的相关函数

In [ ]:
from scipy.spatial import cKDTree


reconstruction_params = {
    "zernike_indices": np.arange(2, 12, dtype=int),
    "max_match_distance_px": 0.45 * derived_params["lenslet_pitch_px"],
    "edge_radius_fraction": 0.98,
}


def match_spot_tables(reference_spots, measurement_spots, max_distance_px):
    if reference_spots.empty or measurement_spots.empty:
        raise ValueError("Reference and measurement spot tables must not be empty.")

    ref_xy = reference_spots[["x_px", "y_px"]].to_numpy(dtype=np.float64)
    meas_xy = measurement_spots[["x_px", "y_px"]].to_numpy(dtype=np.float64)
    dist, meas_index = cKDTree(meas_xy).query(ref_xy, k=1)

    candidates = [
        (float(d), int(ref_i), int(meas_i))
        for ref_i, (d, meas_i) in enumerate(zip(dist, meas_index))
        if np.isfinite(d) and d <= max_distance_px
    ]
    candidates.sort(key=lambda item: item[0])

    used_measurements = set()
    records = []
    for distance_px, ref_i, meas_i in candidates:
        if meas_i in used_measurements:
            continue
        used_measurements.add(meas_i)

        ref = reference_spots.iloc[ref_i]
        meas = measurement_spots.iloc[meas_i]
        dx_px = float(meas["x_px"] - ref["x_px"])
        dy_px = float(meas["y_px"] - ref["y_px"])
        records.append(
            {
                "spot_id": int(ref["spot_id"]) if "spot_id" in ref else int(ref_i),
                "ref_x_px": float(ref["x_px"]),
                "ref_y_px": float(ref["y_px"]),
                "meas_x_px": float(meas["x_px"]),
                "meas_y_px": float(meas["y_px"]),
                "dx_px": dx_px,
                "dy_px": dy_px,
                "distance_px": float(np.hypot(dx_px, dy_px)),
                "match_distance_px": distance_px,
            }
        )

    matches = pd.DataFrame.from_records(records).sort_values("spot_id", ignore_index=True)
    if matches.empty:
        raise ValueError("No spot matches found. Try increasing max_match_distance_px or improving spot detection.")
    return matches


def matched_spot_coordinates_and_slopes(matches, params, origin_px=None):
    pixel_pitch_m = params["camera"]["pixel_pitch_m"]
    magnification = params["relay"]["dm_to_sh_magnification"]
    f_mla = params["microlens_array"]["focal_length_m"]

    if origin_px is None:
        origin_px = (float(matches["ref_x_px"].mean()), float(matches["ref_y_px"].mean()))

    x_m = (matches["ref_x_px"].to_numpy() - origin_px[0]) * pixel_pitch_m / magnification
    y_m = -(matches["ref_y_px"].to_numpy() - origin_px[1]) * pixel_pitch_m / magnification
    slope_x = matches["dx_px"].to_numpy() * pixel_pitch_m / f_mla
    slope_y = -matches["dy_px"].to_numpy() * pixel_pitch_m / f_mla
    return np.column_stack([x_m, y_m]), np.column_stack([slope_x, slope_y]), origin_px


def zernike_at_xy(j, xy_m, aperture_radius_m):
    rho = np.linalg.norm(xy_m, axis=1) / aperture_radius_m
    theta = np.arctan2(xy_m[:, 1], xy_m[:, 0])
    values = np.full(xy_m.shape[0], np.nan, dtype=np.float64)
    inside = rho <= 1.0
    values[inside] = zernike_mode(int(j), rho[inside], theta[inside])
    return values


def build_real_zernike_slope_matrix(zernike_indices, xy_m, aperture_m, wavelength_m):
    aperture_radius_m = aperture_m / 2.0
    eps_m = aperture_m / 2000.0
    columns = []
    for j in zernike_indices:
        dz_dx = (
            zernike_at_xy(j, xy_m + np.array([eps_m, 0.0]), aperture_radius_m)
            - zernike_at_xy(j, xy_m - np.array([eps_m, 0.0]), aperture_radius_m)
        ) / (2.0 * eps_m)
        dz_dy = (
            zernike_at_xy(j, xy_m + np.array([0.0, eps_m]), aperture_radius_m)
            - zernike_at_xy(j, xy_m - np.array([0.0, eps_m]), aperture_radius_m)
        ) / (2.0 * eps_m)
        columns.append(np.concatenate([
            wavelength_m / (2.0 * np.pi) * dz_dx,
            wavelength_m / (2.0 * np.pi) * dz_dy,
        ]))
    return np.column_stack(columns)


def reconstruct_zernike_from_matches(matches, params, reconstruction_params):
    xy_m, slopes, origin_px = matched_spot_coordinates_and_slopes(matches, params)
    aperture_m = params["dm"]["active_aperture_m"]
    radius_m = aperture_m / 2.0
    inside = np.linalg.norm(xy_m, axis=1) <= reconstruction_params["edge_radius_fraction"] * radius_m

    xy_fit = xy_m[inside]
    slopes_fit = slopes[inside]
    if xy_fit.shape[0] < len(reconstruction_params["zernike_indices"]):
        raise ValueError("Not enough matched spots inside the DM aperture for this Zernike fit.")

    A = build_real_zernike_slope_matrix(
        reconstruction_params["zernike_indices"],
        xy_fit,
        aperture_m=aperture_m,
        wavelength_m=params["source"]["wavelength_m"],
    )
    b = np.concatenate([slopes_fit[:, 0], slopes_fit[:, 1]])
    keep_rows = np.all(np.isfinite(A), axis=1) & np.isfinite(b)
    coeffs_rad, *_ = np.linalg.lstsq(A[keep_rows], b[keep_rows], rcond=1e-6)

    residual = A[keep_rows] @ coeffs_rad - b[keep_rows]
    coeff_table = pd.DataFrame(
        {
            "zernike_j": reconstruction_params["zernike_indices"],
            "name": [zernike_names.get(int(j), "unknown") for j in reconstruction_params["zernike_indices"]],
            "coeff_rad": coeffs_rad,
        }
    )
    info = {
        "origin_px": origin_px,
        "matched_spots": int(matches.shape[0]),
        "fit_spots": int(xy_fit.shape[0]),
        "rms_displacement_px": float(np.sqrt(np.mean(matches["distance_px"].to_numpy() ** 2))),
        "rms_slope_residual": float(np.sqrt(np.mean(residual ** 2))),
    }
    return coeff_table, info


def plot_spot_displacements(image, matches, output_png=None):
    image = np.asarray(image)
    vmin, vmax = np.percentile(image, [1.0, 99.8])
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image, cmap="gray", origin="upper", vmin=vmin, vmax=vmax)
    ax.quiver(
        matches["ref_x_px"],
        matches["ref_y_px"],
        matches["dx_px"],
        matches["dy_px"],
        color="red",
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.0025,
    )
    ax.scatter(matches["ref_x_px"], matches["ref_y_px"], s=16, c="cyan", label="reference")
    ax.scatter(matches["meas_x_px"], matches["meas_y_px"], s=16, c="orange", label="measurement")
    ax.set_title(f"Spot displacement vectors: {len(matches)} matches")
    ax.axis("off")
    ax.legend(loc="upper right")
    plt.tight_layout()
    if output_png is not None:
        fig.savefig(output_png, dpi=200)
    plt.show()


def plot_reconstructed_zernike_coeffs(coeff_table, output_png=None):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(coeff_table["zernike_j"], coeff_table["coeff_rad"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Zernike index j")
    ax.set_ylabel("coefficient (rad)")
    ax.set_title("Reconstructed Zernike coefficients from measured spot shifts")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if output_png is not None:
        fig.savefig(output_png, dpi=200)
    plt.show()


匹配质点，计算zernike系数，画图对比

In [ ]:
for name in ["reference_spots", "measurement_spots", "measurement_image"]:
    if name not in globals():
        raise RuntimeError(f"Missing {name}. Run the reference and measurement capture/detection cells first.")

spot_matches = match_spot_tables(
    reference_spots,
    measurement_spots,
    max_distance_px=reconstruction_params["max_match_distance_px"],
)
reconstructed_coeffs, reconstruction_info = reconstruct_zernike_from_matches(
    spot_matches,
    params,
    reconstruction_params,
)

base_path = Path(measurement_image_paths["npy"]).with_suffix("")
matches_csv_path = base_path.with_name(base_path.name + "_matches.csv")
coeff_csv_path = base_path.with_name(base_path.name + "_zernike_coeffs.csv")
quiver_png_path = base_path.with_name(base_path.name + "_displacements.png")
coeff_png_path = base_path.with_name(base_path.name + "_zernike_coeffs.png")

spot_matches.to_csv(matches_csv_path, index=False)
reconstructed_coeffs.to_csv(coeff_csv_path, index=False)
plot_spot_displacements(measurement_image, spot_matches, output_png=quiver_png_path)
plot_reconstructed_zernike_coeffs(reconstructed_coeffs, output_png=coeff_png_path)

print(f"Matched spots: {reconstruction_info['matched_spots']}")
print(f"Fit spots inside DM aperture: {reconstruction_info['fit_spots']}")
print(f"RMS displacement: {reconstruction_info['rms_displacement_px']:.3f} px")
print(f"RMS slope residual: {reconstruction_info['rms_slope_residual']:.3e}")
print(f"Saved matches CSV: {matches_csv_path}")
print(f"Saved coefficients CSV: {coeff_csv_path}")

reconstructed_coeffs
